Dataset: A collection of molecular data (structures, properties, or other relevant data).

CNN Architecture: Using CNNs for learning features from molecular images or graphs.

Training the Model: Training the CNN to predict properties like activity, solubility, toxicity, etc.

Here’s a simplified outline for this task. I'll focus on molecular structure images or representations (like SMILES strings) that can be converted to images for input into a CNN.

Steps:
Preprocessing: Convert molecular data (e.g., SMILES) into images or graphs (using tools like RDKit).

Building CNN: Create a CNN architecture suitable for processing molecular images.

Model Training: Train the model on a dataset of known compounds and their properties.

Evaluation: Evaluate the model performance using metrics such as accuracy, ROC-AUC, etc.

In [ ]:
!pip install rdkit tensorflow keras numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.callbacks import EarlyStopping
import os

# Step 1: Data Preparation and Preprocessing
# We use RDKit to generate molecular images (you could use other representations like graphs or 3D structures)
def mol_to_image(mol):
    """
    Convert an RDKit molecule to an image.
    """
    img = Draw.MolToImage(mol, size=(300, 300))
    return img

# Example: Load a molecule and convert to image
smiles = 'CCO'  # Ethanol SMILES
mol = Chem.MolFromSmiles(smiles)
img = mol_to_image(mol)
img.show()

# Assuming you have a dataset of SMILES strings and their properties (e.g., toxicity, solubility)
# Load the dataset (CSV with columns 'smiles' and 'toxicity')
dataset = pd.read_csv('molecular_data.csv')
dataset['molecule'] = dataset['smiles'].apply(Chem.MolFromSmiles)
dataset['image'] = dataset['molecule'].apply(mol_to_image)

# Save images to disk for loading into the CNN later
if not os.path.exists('images'):
    os.makedirs('images')

for idx, row in dataset.iterrows():
    img_path = f"images/{idx}.png"
    row['image'].save(img_path)

# Step 2: Preprocessing Images for CNN
# Convert the images into a format that can be fed into the CNN
image_paths = [f"images/{idx}.png" for idx in dataset.index]
images = [load_img(img_path, target_size=(300, 300)) for img_path in image_paths]
images_array = np.array([img_to_array(img) for img in images])

# Normalize the images to the range [0, 1]
images_array = images_array / 255.0

# Step 3: Prepare Labels (e.g., toxicity, solubility)
labels = dataset['toxicity'].values  # Assuming 'toxicity' is the target variable

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(images_array, labels, test_size=0.2, random_state=42)

# Step 4: Build the CNN Model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(300, 300, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # For binary classification (toxicity: 0 or 1)

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Step 5: Train the Model
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop])

# Step 6: Evaluate the Model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc * 100:.2f}%")

# Step 7: Visualization (Plotting training history)
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

# Step 8: Make Predictions on New Data
def predict_toxicity(smiles):
    mol = Chem.MolFromSmiles(smiles)
    img = mol_to_image(mol)
    img_array = img_to_array(img.resize((300, 300))) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    return prediction

# Test the model on a new molecule (e.g., Aspirin)
aspirin_smiles = 'CC(=O)OC1=CC=CC=C1C(=O)O'
print(f"Aspirin Toxicity Prediction: {predict_toxicity(aspirin_smiles)}")
